# 补充：不依赖 `langchain-community` 的文档读取方法

官方文档: https://docs.langchain.com/oss/python/integrations/document_loaders

02 笔记讲的 `TextLoader`、`CSVLoader`、`JSONLoader` 都在 `langchain-community` 里。这个包 2026-05-22 起停止维护，官方文档也不再列出这几个加载器（见 02 笔记 2.2 开头）。这份笔记整理现在的做法。

先看官方自己怎么做。官方现在的两个教程都**没有**用加载器类：

* [语义搜索教程](https://docs.langchain.com/oss/python/langchain/knowledge-base)读 PDF：自己写了一个 `load_pdf_pages()` 函数，用 `pypdf` 逐页取出文字、构造 `Document`。`PyPDFLoader` 只在一句话里提到，链接到集成列表。
* [RAG 教程](https://docs.langchain.com/oss/python/deepagents/rag)读文档页面：用 `requests.get()` 取回页面的 Markdown，再构造 `Document`。

也就是说，`Document` 和 `BaseLoader` 都还在（它们属于 `langchain-core`，没有停止维护），变的是**谁来读文件**：

* 简单格式（txt、Markdown、JSON、CSV）：Python 标准库就能读，自己构造 `Document`；
* 复杂格式（PDF、Word、网页）：交给专门的解析工具。这些工具大多有独立维护的 LangChain 集成包，或者能直接输出 Markdown。

![按格式选择读取方法](assets/06-choose-reader.svg)

为什么尽量输出 **Markdown**：Markdown 能保留标题层级和表格，下一步切分时可以按标题切，表格也不会被拆散。PyMuPDF4LLM、Docling、MarkItDown 都输出 Markdown。

本笔记用到的工具（版本为本项目 `.venv` 中安装的版本）：

| 工具 | 安装 | 版本 | 许可证 | 本笔记 |
|---|---|---|---|---|
| Python 标准库（`pathlib`、`json`、`csv`） | 不用装 | — | — | 运行 |
| pypdf | `pypdf` | 6.19.0 | BSD-3-Clause | 运行 |
| PyMuPDF4LLM | `langchain-pymupdf4llm` | 1.28.0 | **AGPL-3.0** | 运行 |
| MarkItDown | `markitdown[docx]` | 0.1.8 | MIT | 运行 |
| Docling | `langchain-docling` + `docling-slim[format-docx]` | 3.0.0 | MIT | 运行 docx；PDF 没有运行 |
| Unstructured | `langchain-unstructured` + `unstructured[md,docx]` | 1.0.1 / 0.27.22 | MIT / Apache-2.0 | 运行 Markdown、docx；PDF 没有运行 |
| requests | 已安装 | 2.34.2 | Apache-2.0 | 运行（会联网） |

本笔记的代码都不调用大模型。

## 1. 简单格式：标准库读取，自己构造 `Document`

`Document` 只有 `page_content` 和 `metadata` 两个常用字段（见 02 笔记）。读出文本、填好这两个字段，就得到了和加载器一样的结果。

### 1.1 txt、Markdown

In [1]:
from pathlib import Path

from langchain_core.documents import Document

path = Path("load/01-langchain-gbk.txt")

doc = Document(
    page_content=path.read_text(encoding="gbk"),   # 编码要自己写清楚
    metadata={"source": str(path)},
)
print(doc)

page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务' metadata={'source': 'load/01-langchain-gbk.txt'}


这和 02 笔记里 `TextLoader(file_path, encoding="gbk").load()[0]` 的结果一样。Markdown 文件的读法完全相同，读出来的文本保留 `#` 标题，后面可以按标题切分。

不知道文件是什么编码时，可以先用 `chardet` 检测（本项目已经安装）。`TextLoader(autodetect_encoding=True)` 内部用的也是它：

In [2]:
import chardet

raw = path.read_bytes()
detected = chardet.detect(raw)
print(detected)

text = raw.decode(detected["encoding"])
print(text[:20])

{'encoding': 'GB18030', 'confidence': 0.834170899663136, 'language': 'zh', 'mime_type': 'text/plain'}
LangChain 是一个用于构建基于大


检测结果是 GB18030，它是 GBK 的超集，用它解码 GBK 文件没有问题。检测结果带有可信度 `confidence`，并不保证正确；编码已知时，直接写明编码更可靠。

### 1.2 JSON

JSON 用 `json.loads` 读成 Python 的字典和列表，再按普通 Python 数据处理。`JSONLoader` 里 `jq_schema`、`content_key`、`metadata_func` 做的事，这里都是一行 Python：

In [3]:
import json

path = Path("load/03-load.json")
data = json.loads(path.read_text(encoding="utf-8"))

docs = [
    Document(
        page_content=message["content"],               # 相当于 content_key="content"
        metadata={                                     # 相当于 metadata_func
            "source": str(path),
            "sender": message["sender"],
            "timestamp": message["timestamp"],
        },
    )
    for message in data["messages"]                    # 相当于 jq_schema=".messages[]"
]

for doc in docs:
    print(doc)

page_content='Hello, how are you today?' metadata={'source': 'load/03-load.json', 'sender': 'Alice', 'timestamp': '2023-05-15T10:00:00'}
page_content='I'm doing well, thanks for asking!' metadata={'source': 'load/03-load.json', 'sender': 'Bob', 'timestamp': '2023-05-15T10:02:00'}
page_content='Would you like to meet for lunch?' metadata={'source': 'load/03-load.json', 'sender': 'Alice', 'timestamp': '2023-05-15T10:05:00'}
page_content='Sure, that sounds great!' metadata={'source': 'load/03-load.json', 'sender': 'Bob', 'timestamp': '2023-05-15T10:07:00'}


嵌套更深的数据也一样，直接用下标取。`03-response.json` 是一个分页接口的返回结果，文章列表在 `data.items` 里，作者信息又嵌套了一层：

In [4]:
path = Path("load/03-response.json")
data = json.loads(path.read_text(encoding="utf-8"))

docs = [
    Document(
        page_content=f"{item['title']}\n{item['content']}",   # 标题和正文一起作为内容
        metadata={
            "source": str(path),
            "id": item["id"],
            "author": item["author"]["name"],                 # 嵌套字段，直接取
            "created_at": item["created_at"],
        },
    )
    for item in data["data"]["items"]
]

print(len(docs))
print(docs[0])

3
page_content='Understanding JSONLoader
This article explains how to parse API responses...' metadata={'source': 'load/03-response.json', 'id': 101, 'author': 'Alice', 'created_at': '2023-10-05T08:12:33Z'}


如果确实要把整个对象转成字符串（02 笔记情况1 的做法），用 `json.dumps(obj, ensure_ascii=False)`。这样中文不会被转义成 `\uXXXX`，这正是 `JSONLoader(text_content=False)` 改不了的地方。

### 1.3 CSV

`csv.DictReader` 把每一行读成字典，键是表头。每行构造一个 `Document`，哪些列放内容、哪些列放元数据，自己决定：

In [5]:
import csv

path = Path("load/02-load.csv")

with path.open(encoding="utf-8", newline="") as f:
    docs = [
        Document(
            page_content=f"{row['title']}\n{row['content']}",
            metadata={"source": str(path), "row": i, "id": row["id"], "author": row["author"]},
        )
        for i, row in enumerate(csv.DictReader(f))
    ]

print(len(docs))
print(docs[0])

4
page_content='Introduction to Python
Python is a popular programming language.' metadata={'source': 'load/02-load.csv', 'row': 0, 'id': '1', 'author': 'John Doe'}


### 1.4 需要统一的接口时：继承 `BaseLoader`

上面的写法够用了。但如果一个项目要读很多文件、很多种格式，最好让每种读取方式都有同样的 `load()` 和 `lazy_load()`，后面的代码就不用关心文件是什么格式。这时可以自己写一个 `BaseLoader` 的子类。

按 02 笔记讲的 `BaseLoader` 源码，子类**只需要实现 `lazy_load()`**，用 `yield` 每次产出一个 `Document`；`load()`、`aload()` 等方法都由 `BaseLoader` 提供。下面写一个通用的 JSON 加载器，用一个函数代替 `jq_schema` 去取记录列表：

In [6]:
from collections.abc import Callable, Iterable, Iterator, Sequence

from langchain_core.document_loaders import BaseLoader


class JSONRecordLoader(BaseLoader):
    """把 JSON 文件中的每条记录读成一个 Document。"""

    def __init__(
        self,
        file_path: str | Path,
        get_records: Callable[[object], Iterable[dict]],
        content_key: str,
        metadata_keys: Sequence[str] = (),
    ) -> None:
        self.file_path = Path(file_path)
        self.get_records = get_records        # 从整个 JSON 中取出记录列表，代替 jq_schema
        self.content_key = content_key        # 哪个字段作为 page_content
        self.metadata_keys = metadata_keys    # 哪些字段放进 metadata

    def lazy_load(self) -> Iterator[Document]:
        data = json.loads(self.file_path.read_text(encoding="utf-8"))
        for i, record in enumerate(self.get_records(data)):
            metadata = {"source": str(self.file_path), "index": i}
            metadata.update({key: record[key] for key in self.metadata_keys})
            yield Document(page_content=record[self.content_key], metadata=metadata)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
loader = JSONRecordLoader(
    file_path="load/03-response.json",
    get_records=lambda data: data["data"]["items"],
    content_key="content",
    metadata_keys=["id", "title", "created_at"],
)

docs = loader.load()             # BaseLoader 提供的 load()：list(self.lazy_load())
print(len(docs))
print(docs[0])

for doc in loader.lazy_load():   # 也可以逐个取
    print(doc.metadata["id"], doc.page_content)

3
page_content='This article explains how to parse API responses...' metadata={'source': 'load/03-response.json', 'index': 0, 'id': 101, 'title': 'Understanding JSONLoader', 'created_at': '2023-10-05T08:12:33Z'}
101 This article explains how to parse API responses...
102 Learn to handle nested structures with...
103 Best practices for preserving metadata...


同一个类换一个 `get_records`，就能读 `03-load.json`：`get_records=lambda data: data["messages"]`。

定义类的那个单元格打印了一行 `[transformers] PyTorch was not found...`。原因是 `BaseLoader` 所在的模块会导入 `langchain-text-splitters`（`load_and_split()` 默认要用其中的 `RecursiveCharacterTextSplitter`），而 `langchain-text-splitters` 又会导入 `transformers`。本项目装了 `transformers`，没装 PyTorch，所以打印这条提示，不影响使用。

## 2. PDF

PDF 要按类型选工具：

| PDF 的类型 | 工具 | 小节 |
|---|---|---|
| 文字型，只要文本 | pypdf | 2.1 |
| 文字型，要保留标题、表格 | PyMuPDF4LLM | 2.2 |
| 扫描件（图片）、复杂版面 | Docling、Unstructured 等，需要 OCR 和版面分析模型 | 2.3 |

下面都用 `load/04-sample.pdf`：中国科学院国家天文台 2023 年部门预算，36 页，第 7 页（下标 6）是一张“收支总表”，正好用来比较各工具怎样处理表格。

### 2.1 pypdf：官方教程的写法

下面的函数和官方语义搜索教程里的 `load_pdf_pages()` 一样：每页生成一个 `Document`，`metadata` 记下文件和页码（从 0 开始）：

In [8]:
import pypdf


def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


docs = load_pdf_pages("load/04-sample.pdf")
print(len(docs))
print(docs[6].metadata)
print(docs[6].page_content.rstrip())   # 去掉页尾的空行

36
{'source': 'load/04-sample.pdf', 'page': 6}
 
4 
收支总表 
   部门公开表 1 
单位：万元 
收      入 支      出 
项    目 预算数 项    目 预算数 
一、一般公共预算拨款收入 57,806.77 一、科学技术支出 143,981.03 
二、政府性基金预算拨款收入  二、社会保障和就业支出 1,930.68 
三、国有资本经营预算拨款  三、住房保障支出 1,314.37 
四、事业收入 32,000.00   
五、事业单位经营收入    
六、其他收入 2,200.00   
     
本年收入合计 92,006.77 本年支出合计 147,226.08 
使用非财政拨款结余  结转下年 50,997.08 
上年结转 106,216.39   
收 入 总 计 198,223.16 支 出 总 计 198,223.16


文字都取出来了，但表格被**拉平**了：左右两栏挤在同一行（“项 目 预算数 项 目 预算数”），空单元格直接消失，看不出哪个数字属于哪一列。页码“4”也混在正文里。

另外，`extract_text()` 只能取出 PDF 里本来就有的文字。扫描件的每一页都是图片，取出来是空字符串，这时需要 OCR（见 2.3）。

`langchain-community` 的 `PyPDFLoader` 内部用的也是 pypdf，效果相同。

### 2.2 PyMuPDF4LLM：输出 Markdown

`langchain-pymupdf4llm` 是 PyMuPDF 官方维护的集成包，列在 LangChain 官方的文档加载器页面上。它把每页转成 Markdown，标题变成 `#`，表格变成 Markdown 表格。

> ⚠️ 许可证是 **AGPL-3.0**：把用到它的程序分发给别人，或者作为网络服务给别人使用时，要按 AGPL 公开整个程序的源代码；不想公开，就要购买 Artifex 的商业许可。个人学习不受影响。

In [1]:
import os

import pymupdf
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

pymupdf.set_messages(path=os.devnull)   # 关掉每页都会打印一次的解析提示

loader = PyMuPDF4LLMLoader("load/04-sample.pdf", mode="page")
docs = loader.load()

print(len(docs))
print(type(docs[6]))
print({k: docs[6].metadata[k] for k in ["source", "page", "total_pages", "title"]})
print(docs[6].page_content.rstrip())

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


36
<class 'langchain_core.documents.base.Document'>
{'source': 'load/04-sample.pdf', 'page': 6, 'total_pages': 36, 'title': '中国科学院国家天文台2023年度部门预算'}
# 收支总表 

部门公开表 1 <u>单位：万元</u> 

|收 入||支 出||
|---|---|---|---|
|项 目|预算数|项 目|预算数|
|一、一般公共预算拨款收入|57,806.77|一、科学技术支出|143,981.03|
|二、政府性基金预算拨款收入||二、社会保障和就业支出|1,930.68|
|三、国有资本经营预算拨款||三、住房保障支出|1,314.37|
|四、事业收入|32,000.00|||
|五、事业单位经营收入||||
|六、其他收入|2,200.00|||
|<br>本年收入合计|92,006.77|本年支出合计|147,226.08|
|使用非财政拨款结余||结转下年|50,997.08|
|上年结转|106,216.39|||
|收 入 总 计|**198,223.16**|支 出 总 计|**198,223.16**|



4


同一页的结果：“收支总表”变成了 `#` 标题，表格保留了四列，空单元格也留着位置，`198,223.16` 的加粗也保留了下来。

`PyMuPDF4LLMLoader` 的参数（`langchain_pymupdf4llm/pymupdf4llm_loader.py`）：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `file_path` | 必填 | 本地路径或 URL |
| `mode` | `"page"` | `"page"`：每页一个 `Document`；`"single"`：整份 PDF 一个 `Document` |
| `pages_delimiter` | `"\n-----\n\n"` | `mode="single"` 时，页与页之间插入的分隔符 |
| `password` | `None` | 加密 PDF 的密码 |
| `use_layout` | `True` | 是否先做版面分析（识别标题、表格、正文）。关掉后速度更快，但结构识别会差一些；本机装了 Tesseract 时，没有文字的页面会自动 OCR |
| `extract_images` / `images_parser` | `False` / `None` | 是否用 `images_parser` 把 PDF 里的图片解析成文字 |
| `**pymupdf4llm_kwargs` | — | 其余参数原样传给 `pymupdf4llm.to_markdown()` |

返回值：`mode="page"` 时每页一个 `Document`，`metadata` 包含 `source`、`page`（从 0 开始）、`total_pages`，以及 PDF 自带的 `title`、`author`、创建时间等信息。

### 2.3 扫描件、复杂版面：Docling、Unstructured（没有运行）

扫描件和版面复杂的 PDF（多栏、跨页表格、公式）要用 OCR 和版面分析模型。官方页面上下载量最大的两个是 Docling 和 Unstructured，但它们在本地处理 PDF 都要装 PyTorch：

| 工具 | 本地处理 PDF 需要安装 | 说明 |
|---|---|---|
| Docling | `langchain-docling[local]`（会多装 25 个包，包括 torch、torchvision） | IBM 开源，MIT 许可。输出 Markdown，或直接输出切好的块 |
| Unstructured | `unstructured[pdf]`（30 个包，包括 torch），再用 Homebrew 装 poppler、tesseract 等系统依赖 | 也可以不在本地处理，改用 Unstructured 的云端 API（需要 API key） |

所以本笔记只在 3.2、3.3 节用它们处理 docx、Markdown（不需要模型），PDF 的写法如下，没有运行：

```python
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

docs = DoclingLoader(file_path="load/04-sample.pdf", export_type=ExportType.MARKDOWN).load()
```

```python
from langchain_unstructured import UnstructuredLoader

docs = UnstructuredLoader("load/04-sample.pdf", strategy="hi_res").load()   # hi_res：用版面分析模型
```

> ⚠️ **与官方文档不同**：LangChain 官方的 Docling 页面写的安装命令是 `pip install -qU langchain-docling`。但 `langchain-docling` 从 3.0.0 起，默认只装连接 Docling 服务的轻量客户端；想在本地转换文件，要安装 `langchain-docling[local]`（见它在 PyPI 上的说明）。本项目装的就是不带 `[local]` 的 3.0.0，所以连 docx 都要另外装 `docling-slim[format-docx]` 才能处理。

中文扫描件还可以看官方页面上列出的 [MinerU](https://mineru.net) 和 [PaddleOCR-VL](https://www.paddleocr.com)，它们对中文版面和表格的识别比较好。

## 3. Word、HTML 等其他格式

### 3.1 MarkItDown：转成 Markdown 再构造 `Document`

MarkItDown 是微软开源的工具，能把 Word、PowerPoint、Excel、HTML、PDF 等转成 Markdown。它不在 LangChain 官方的加载器列表里（有一个第三方写的 `langchain-markitdown`），直接用它转换、自己构造 `Document` 就行：

In [10]:
from markitdown import MarkItDown

converter = MarkItDown()

docs = []
for path in ["load/05-sgg_chat.docx", "load/07-load.html"]:
    result = converter.convert(path)         # 返回 DocumentConverterResult
    docs.append(
        Document(
            page_content=result.markdown,     # 转换后的 Markdown
            metadata={"source": path, "title": result.title},
        )
    )

for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:600])
    print("=" * 60)

{'source': 'load/05-sgg_chat.docx', 'title': None}
**尚硅谷师资团队评估报告**

**一、机构基本信息**

**名称**：尚硅谷教育科技有限公司
**总部地址**：北京市海淀区中关村软件园创新大厦B座5层
**成立日期**：2013年3月15日
**法定代表人**：张明哲
**注册资本**：人民币2000万元
**员工规模**：全职讲师150人，技术助教300人
**官方电话**：010-66669999
**官方邮箱**：[hr@atguigu.com](https://mailto:hr@atguigu.com/)

**二、师资团队核心优势**

1. **技术背景深厚**
   * 90%讲师来自BAT、字节跳动等一线互联网企业，平均技术从业年限8年+
   * 35%讲师拥有开源项目贡献经历（如Apache、Linux基金会项目）
2. **教学能力突出**
   * 所有讲师均通过“教学能力三级认证体系”（技术能力、课程设计、课堂表达）
   * 学员平均评分4.9/5.0（2023年内部调研数据）
3. **行业影响力显著**
   * 12位讲师担任Oracle/Red Hat/AWS认证官方考官
   * 出版技术书籍28本，累计销量超50万册

**三、师资结构分析**

| **类别** | **人数** | **占比** | **典型代表** |
| --- | --- | --- | --- |
| 
{'source': 'load/07-load.html', 'title': 'RAG:将检索与生成方式相结合来做生成任务'}
首发于自然语言处理算法与实践
切换模式

# RAG:将检索与生成方式相结合来做生成任务

烛之文

## 1、前言

在上一篇<kNN-NER：利用knn近邻算法来做命名实体识别>提及到文中提出kNN-NER框架是一种检索式增强的方法（retrieval augmented methods），就去查看有关retrieval augmented的paper，了解其核心思想，觉得检索式增强的方法很适合许多业务场景使用，因其以一种简捷的方式将外部知识融于模型中去。今天就分享一篇来自Facebook AI Research的paper<Retrieval-Augm

* Word 里的加粗、列表、表格都转成了 Markdown 语法，表格保留了行列结构；
* HTML 的 `<title>` 被取出来放在 `result.title` 里，正文的标题变成了 `#`、`##`。

MarkItDown 处理 PDF 时调用 pdfminer，整份 PDF 输出成一个字符串，不分页，所以 PDF 还是优先用第 2 节的方法。

### 3.2 Docling：处理 docx

Docling 处理 Word 不需要模型，装了 `docling-slim[format-docx]` 就能运行。`export_type` 决定输出形式：

In [2]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

loader = DoclingLoader(file_path="load/05-sgg_chat.docx", export_type=ExportType.MARKDOWN)
docs = loader.load()

print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:900])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


1
{'source': 'load/05-sgg_chat.docx'}
**尚硅谷师资团队评估报告**

**一、机构基本信息**

**名称** ：尚硅谷教育科技有限公司 **总部地址** ：北京市海淀区中关村软件园创新大厦B座5层 **成立日期** ：2013年3月15日 **法定代表人** ：张明哲 **注册资本** ：人民币2000万元 **员工规模** ：全职讲师150人，技术助教300人 **官方电话** ：010-66669999 **官方邮箱** ： [hr@atguigu.com](https://mailto:hr@atguigu.com/)

**二、师资团队核心优势**

1. **技术背景深厚**
    - 90%讲师来自BAT、字节跳动等一线互联网企业，平均技术从业年限8年+
    - 35%讲师拥有开源项目贡献经历（如Apache、Linux基金会项目）
2. **教学能力突出**
    - 所有讲师均通过“教学能力三级认证体系”（技术能力、课程设计、课堂表达）
    - 学员平均评分4.9/5.0（2023年内部调研数据）
3. **行业影响力显著**
    - 12位讲师担任Oracle/Red Hat/AWS认证官方考官
    - 出版技术书籍28本，累计销量超50万册

**三、师资结构分析**

| **类别**     |   **人数** | **占比**   | **典型代表**             |
|--------------|------------|------------|--------------------------|
| 架构师级讲师 |         25 | 16.7%      | 前阿里P9分布式系统专家   |
| 高级研发讲师 |         80 | 53.3%      | 腾讯T4级后台开发工程师   |
| 新兴领域讲师 |         45 | 30%        | 原字节AIGC项目组核心成员 |

**教学方向覆盖** ：

- 后端开发（Java/Go/Python）
- 大数据（Hadoop/Spark/Flin


`ExportType.MARKDOWN` 时，每个文件生成一个 `Document`，内容是 Markdown，表格对齐得很整齐。开头那行 `The plugin langchain_docling will not be loaded...` 是 Docling 的提示：它默认不加载外部插件，不影响转换。

`export_type` 的默认值是 `ExportType.DOC_CHUNKS`：Docling 先按文档结构**自己切好块**，每块一个 `Document`。切块时要计算 token 数，第一次运行会从 Hugging Face 下载一个分词器（需要联网，所以下面会打印 Hugging Face 的提示）：

In [3]:
chunks = DoclingLoader(file_path="load/05-sgg_chat.docx").load()   # 默认 ExportType.DOC_CHUNKS

print(len(chunks))
print(chunks[2].page_content)
print(chunks[2].metadata)

7
架构师级讲师, **人数** = 25. 架构师级讲师, **占比** = 16.7%. 架构师级讲师, **典型代表** = 前阿里P9分布式系统专家. 高级研发讲师, **人数** = 80. 高级研发讲师, **占比** = 53.3%. 高级研发讲师, **典型代表** = 腾讯T4级后台开发工程师. 新兴领域讲师, **人数** = 45. 新兴领域讲师, **占比** = 30%. 新兴领域讲师, **典型代表** = 原字节AIGC项目组核心成员
**教学方向覆盖**
：
- 后端开发（Java/Go/Python）
- 大数据（Hadoop/Spark/Flink）
- 人工智能（机器学习/深度学习）
- 云原生（K8s/Service Mesh）
{'source': 'load/05-sgg_chat.docx', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/tables/0', 'parent': {'$ref': '#/body'}, 'children': [{'$ref': '#/groups/5'}, {'$ref': '#/groups/6'}, {'$ref': '#/groups/7'}, {'$ref': '#/groups/8'}], 'content_layer': 'body', 'label': 'table', 'prov': []}, {'self_ref': '#/texts/37', 'parent': {'$ref': '#/groups/9'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}, {'self_ref': '#/texts/38', 'parent': {'$ref': '#/groups/9'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}, {'self_ref': '#/texts/39', 'parent

这一块包含 Word 里的“师资结构分析”表格和后面的“教学方向覆盖”列表。表格部分，Docling 把每个单元格写成“行名, 列名 = 值”的句子，这样一行数据单独被检索出来时，也知道每个数字是什么意思。这些块的 `metadata` 里还有一个很大的 `dl_meta` 字段，记录块在原文档中的位置。

DOC_CHUNKS 模式等于把“加载”和“切分”合成了一步。想自己控制切分（下一节的内容），就用 MARKDOWN 模式。

### 3.3 Unstructured：按元素拆分

`langchain-unstructured` 的 `UnstructuredLoader` 和前面的工具都不同：它把文档拆成一个个**元素**（标题、段落、列表项、表格……），每个元素一个 `Document`，元素类型记在 `metadata["category"]` 里。默认 `partition_via_api=False`，在本地处理：

In [13]:
from langchain_unstructured import UnstructuredLoader

docs = UnstructuredLoader("load/06-load.md").load()

print(len(docs))
for doc in docs[:6]:
    print(f"{doc.metadata['category']:<18}", repr(doc.page_content[:40]))

19
Title              '自然语言处理技术文档'
UncategorizedText  '本文档用于测试UnstructuredMarkdownLoader的中文处理能力'
Title              '第一章：简介'
UncategorizedText  '自然语言处理(NLP)是人工智能的重要分支，主要技术包括：'
ListItem           '文本分类'
ListItem           '命名实体识别'


一个 400 多字的 Markdown 文件被拆成了 19 个 `Document`。这种粒度对检索来说通常太细，可以传 `chunking_strategy="by_title"`，让 Unstructured 按标题把元素合并成块（这个参数原样传给 `unstructured` 的 `partition` 函数，默认每块最多 500 个字符）。

上面的 `libmagic is unavailable` 警告是说本机没有装 libmagic（`brew install libmagic`），Unstructured 改用文件扩展名判断文件类型，不影响结果。

## 4. 网页：requests 取回，再转成 Markdown

官方 RAG 教程读 LangChain 文档时，直接请求页面的 `.md` 版本（LangChain 文档站给每个页面都提供 Markdown 版本），取回的文本就是 `page_content`：

In [14]:
import requests

url = "https://docs.langchain.com/oss/python/langchain/retrieval.md"
response = requests.get(url, timeout=20)
response.raise_for_status()

doc = Document(page_content=response.text, metadata={"source": url})
print(len(doc.page_content))
print(doc.page_content[:300])

15548
> ## Documentation Index
> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Retrieval

Large Language Models (LLMs) are powerful, but they have two key limitations:

* **Finite context**: they 


大多数网站没有 `.md` 版本，取回的是 HTML，再转成 Markdown 就行：`MarkItDown().convert(url)` 可以直接传网址；也可以先用 `requests` 取回 HTML，再用 `markdownify`（MarkItDown 依赖它，已经装上了）转换。需要渲染 JavaScript、批量爬取整站时，官方页面 Webpages 一类列出了 Firecrawl、Apify 等服务。

## 5. 小结

| 场景 | 推荐做法 | `page_content` | 学习重点 |
|---|---|---|---|
| txt、Markdown、JSON、CSV | 标准库读取，自己构造 `Document` | 原文本 | ⭐⭐ 必须会 |
| 统一多种读取方式 | 继承 `BaseLoader`，只实现 `lazy_load()` | 由自己决定 | ⭐⭐ 必须会 |
| 文字型 PDF | pypdf 逐页读取 | 纯文本，表格被拉平 | ⭐ 会用 |
| 要保留结构的 PDF | PyMuPDF4LLM | Markdown | ⭐ 会用，注意 AGPL |
| 扫描件、复杂版面 | Docling、Unstructured（需 PyTorch），中文可看 MinerU | Markdown 或元素 | 了解 |
| Word、HTML 等 | MarkItDown；Docling；Unstructured | Markdown 或元素 | 了解 |
| 网页 | requests + 转 Markdown | Markdown | 了解 |

不管用哪种工具，最后都要得到 `list[Document]`：`page_content` 尽量是保留了结构的文本，`metadata` 记下来源、页码等信息。下一步就是把这些 `Document` 交给文本切分器。